In [ ]:
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import numpy as np
from sklearn import preprocessing
from sklearn import utils
from sklearn.metrics import mean_squared_error
from math import sqrt
from sklearn.metrics import r2_score

import matplotlib.pyplot as plt
import seaborn as sns

#from sklearn.preprocessing import MinMaxScaler
#scaler = MinMaxScaler()
#import pickle
#from sklearn.externals import joblib

In [ ]:
#fpath = 'E:/DOCTOR/SB/ML/data/ML/'
fpath = '../data/CRU/station/'
#fpath = '../ML/'

stmm = 1
enmm = 12
mon = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
       'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for mm in np.arange(stmm,enmm+1):
    print(mm)
    train = pd.read_csv(fpath + repr(mm) + '_train_new.csv')
    print(train.shape)
    train = train.set_index("STN") # STN을 Index로 지정
    train_year = train["YEAR"]
    train = train[[ "LAT", "LON",
                   "NDVI_t", "LST_DAY_t", "LST_NIGHT_t",
                   "NDVI_a", "LST_DAY_a", "LST_NIGHT_a",
                   "Water", "Evergreen_trees", "Decidous_trees",
                   "Shrub", "Grass", "crops", "Urban", "Snow",
                   "Barren", "Unclassifiend", "SAT"]]

    test = pd.read_csv(fpath + repr(mm) + '_test_new.csv')
    print(test.shape)
    test = test.set_index("STN") # STN을 Index로 지정
    test_year = test["YEAR"]
    test = test[[ "LAT", "LON", 
                 "NDVI_t", "LST_DAY_t", "LST_NIGHT_t",
                 "NDVI_a", "LST_DAY_a", "LST_NIGHT_a",
                 "Water", "Evergreen_trees", "Decidous_trees",
                 "Shrub", "Grass", "crops", "Urban", "Snow",
                 "Barren", "Unclassifiend", "SAT"]]

    
    label_name = "SAT"
    feature_names = train.columns.difference([label_name])

# Train
    X_train = train[feature_names]
    y_train = train[label_name] 
    X_test = test[feature_names]
    y_test = test[label_name]

    
#################
# Random Forest #
#################


# range함수를 이용하여 n_estimators_list에 10부터 200까지 10단위로 자연수를 생성합니다.
    n_estimators_list = range(1000, 1001, 1)

# 각각의 n_estimators에 따른 결과를 저장하기 위한 리스트(history)를 만듭니다.
    history = []
    

# n_estimators_list에 있는 숫자만큼 돌릴 for문을 선언합니다.
    for n_estimators in n_estimators_list:
    
        RF = RandomForestRegressor(n_estimators = n_estimators,  
                                       n_jobs = -1,
                                       random_state = 42)
    
# 생성한 model을 학습(fitting)합니다. train 데이터의 feature(X_train)와 label(y_train)을 집어넣습니다.
        RF.fit(X_train, y_train)
# X_train, X_test를 입력으로 predict(예측)한 결과를 각각 y_train_predict, y_test_predict라는 이름의 변수에 할당합니다.
        y_train_predict = RF.predict(X_train)
        y_test_predict = RF.predict(X_test)
        
# 예측값(y_train_predict, y_test_predict)이 얼마나 정답값(y)을 잘 맞췄는지 정확도를 계산합니다.
        train_score = (y_train == y_train_predict).mean()
        test_score = (y_test == y_test_predict).mean()

        train_r2_score = r2_score(y_train, y_train_predict)
        test_r2_score = r2_score(y_test, y_test_predict)

        train_rms = sqrt(mean_squared_error(y_train, y_train_predict))
        test_rms = sqrt(mean_squared_error(y_test, y_test_predict))

        importance = RF.feature_importances_
        
# n_estimators에 따른 train, test 데이터의 각각의 예측 정확도를 출력해봅니다.
        print(f"n_estimators = {n_estimators}, train = {train_score:.6f}, test = {test_score:.6f}")

        print(f"train = {train_r2_score:.6f}, test = {test_r2_score:.6f}")
        print(f"train = {train_rms:.6f}, test = {test_rms:.6f}")    
        print(feature_names, importance)
# n_estimators에 따른 train, test 데이터에 대한 예측 정확도를 history에 저장합니다.
        history.append({
            'n_estimators': n_estimators,
            'train': train_score,
            'test': test_score,
            'variable': feature_names,
            'importance': importance,
        })

# 결과를 판다스의 DataFrame형태로 변형하여 출력합니다.
    history = pd.DataFrame(history)
    history

    train.loc[:,'SAT_estimate_RF'] = y_train_predict
    test.loc[:,'SAT_estimate_RF'] = y_test_predict
    train.loc[:,'YEAR'] = train_year
    test.loc[:,'YEAR'] = test_year
    train.loc[:,'bias_RF'] = train['SAT_estimate_RF'] - train['SAT']
    test.loc[:,'bias_RF'] = test['SAT_estimate_RF'] - test['SAT']
    
#    xint = max(test['SAT_estimate_RF'])-min(test['SAT_estimate_RF'])
#    yint = max(test['SAT'])-min(test['SAT'])

#    xmin = min(test['SAT_estimate_RF'])-xint/10.
#    xmax = max(test['SAT_estimate_RF'])+xint/10.
#    ymin = min(test['SAT'])-yint/10.
#    ymax = max(test['SAT'])+yint/10.

#    mmin = min(xmin, ymin)
#    mmax = max(xmax, ymax)
#    mint = mmax - mmin
    
    xint = 25-(-55)
    yint = 25-(-55)

    xmin = -55-xint/10.
    xmax = 25+xint/10.
    ymin = -55-yint/10.
    ymax = 25+yint/10.

    mmin = min(xmin, ymin)
    mmax = max(xmax, ymax)
    mint = mmax - mmin

    plt.figure(figsize = (8, 8))
#    plt.xlim(xmin,xmax)
#    plt.ylim(ymin,ymax)
    plt.xlim(mmin,mmax)
    plt.ylim(mmin,mmax)

#    plt.title('Random Forest ('+ str(mon[mm-1]) + ')', fontsize=20)
    plt.xlabel('Estimated SAT [$^\circ$C]', fontsize=16)
    plt.ylabel('Station SAT [$^\circ$C]', fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
#    plt.scatter(y_test_predict,y_test)

    sns.regplot(x=test['SAT_estimate_RF'],
                y=test['SAT'],
                scatter_kws={'color': 'white'})

#    plt.title('Random Forest (' + str(mon[mm-1]) + ')', fontsize=20)
    plt.xlabel('Estimated SAT [$^\circ$C]', fontsize=16)
    plt.ylabel('Station SAT [$^\circ$C]', fontsize=16)
#    plt.scatter(y_test_predict,y_test)
#    plt.text((xmax+xmin)/2.+xint/3.2,(ymax+ymin)/2.-yint/2.0, f"RMSE = {test_rms:.3f}", fontsize=10)
#    plt.text((xmax+xmin)/2.+xint/2.8,(ymax+ymin)/2.-yint/2.2, f"$R^{2} = {test_r2_score:.3f}$", fontsize=10)
    plt.text((mmax+mmin)/2.+mint/3.2,(mmax+mmin)/2.-mint/2.3, f"RMSE = {test_rms:.3f}", fontsize=10)
    plt.text((mmax+mmin)/2.+mint/2.85,(mmax+mmin)/2.-mint/2.5, f"$R^{2} = {test_r2_score:.3f}$", fontsize=10)
    z = np.polyfit(test["SAT_estimate_RF"], test["SAT"],1)
#    if z[1]<0 :
#        plt.text((xmax+xmin)/2.+xint/8.,(ymax+ymin)/2.-yint/8.,f"$y = {z[0]:.3f}x {z[1]:.3f}$", fontsize=10)
#    else:
#        plt.text((xmax+xmin)/2.+xint/8.,(ymax+ymin)/2.-yint/8.,f"$y = {z[0]:.3f}x +{z[1]:.3f}$", fontsize=10)
    if z[1]<0 :
        plt.text((mmax+mmin)/2.+mint/8.,(mmax+mmin)/2.-mint/8.,f"$y = {z[0]:.3f}x {z[1]:.3f}$", fontsize=10)
    else:
        plt.text((mmax+mmin)/2.+mint/8.,(mmax+mmin)/2.-mint/8.,f"$y = {z[0]:.3f}x +{z[1]:.3f}$", fontsize=10)
        
    sns.scatterplot(x='SAT_estimate_RF',
                    y='SAT',
 #                   hue='YEAR', # different colors by group
 #                   style='STN', # different shapes by group
                    s=10, # marker size
                    data=test, legend = False,
                    color='k')

    #plt.show()
    plt.savefig(fpath + 'FIG/eps/' + repr(mm) + '_rf_reg.eps')
    plt.savefig(fpath + 'FIG/ML/' + repr(mm) + '_rf_reg.png')
   


#################
### LightGBM ###
#################
    n_estimators_list = range(1000, 1001, 1)
    history = []

    for n_estimators in n_estimators_list:
    
        LGBM = LGBMRegressor(n_estimators = n_estimators,  
                                          learning_rate = 0.1)#,
#                                          max_depth = 6)
    
        LGBM.fit(X_train, y_train)
        y_train_predict = LGBM.predict(X_train)
        y_test_predict = LGBM.predict(X_test)

        train_score = (y_train == y_train_predict).mean()
        test_score = (y_test == y_test_predict).mean()

        train_r2_score = r2_score(y_train, y_train_predict)
        test_r2_score = r2_score(y_test, y_test_predict)

        train_rms = sqrt(mean_squared_error(y_train, y_train_predict))
        test_rms = sqrt(mean_squared_error(y_test, y_test_predict))

        importance = LGBM.booster_.feature_importance(importance_type="gain")
        importance = importance / sum(importance)
#        i_min = importance.min()
#        i_max = importance.max()
#        i_denom = i_max-i_min
#        for i in range(len(importance)):
#            importance[i] = float(importance[i] - i_min) / float(i_denom)
#        importance[:] =scaler.fit_transform(importance[:])
        print(f"n_estimators = {n_estimators}, train = {train_score:.6f}, test = {test_score:.6f}")

        print(f"train = {train_r2_score:.6f}, test = {test_r2_score:.6f}")
        print(f"train = {train_rms:.6f}, test = {test_rms:.6f}")    
        print(feature_names, importance)

        history.append({
            'n_estimators': n_estimators,
            'train': train_score,
            'test': test_score,
            'variable': feature_names,
            'importance': importance,
        })

# 결과를 판다스의 DataFrame형태로 변형하여 출력합니다.
    history = pd.DataFrame(history)
    history


    train.loc[:,'SAT_estimate_LGBM'] = y_train_predict
    test.loc[:,'SAT_estimate_LGBM'] = y_test_predict
    train.loc[:,'YEAR'] = train_year
    test.loc[:,'YEAR'] = test_year
    train.loc[:,'bias_LGBM'] = train['SAT_estimate_LGBM'] - train['SAT']
    test.loc[:,'bias_LGBM'] = test['SAT_estimate_LGBM'] - test['SAT']

#    xint = max(test['SAT_estimate_LGBM'])-min(test['SAT_estimate_LGBM'])
#    yint = max(test['SAT'])-min(test['SAT'])

#    xmin = min(test['SAT_estimate_LGBM'])-xint/10.
#    xmax = max(test['SAT_estimate_LGBM'])+xint/10.
#    ymin = min(test['SAT'])-yint/10.
#    ymax = max(test['SAT'])+yint/10.

#    mmin = min(xmin, ymin)
#    mmax = max(xmax, ymax)
#    mint = mmax - mmin

    xint = 25-(-55)
    yint = 25-(-55)

    xmin = -55-xint/10.
    xmax = 25+xint/10.
    ymin = -55-yint/10.
    ymax = 25+yint/10.

    mmin = min(xmin, ymin)
    mmax = max(xmax, ymax)
    mint = mmax - mmin
    
    plt.figure(figsize = (8, 8))
#    plt.xlim(xmin,xmax)
#    plt.ylim(ymin,ymax)
    plt.xlim(mmin,mmax)
    plt.ylim(mmin,mmax)

#    plt.title('LightGBM ('+ str(mon[mm-1]) + ')', fontsize=20)
    plt.xlabel('Estimated SAT [$^\circ$C]', fontsize=16)
    plt.ylabel('Station SAT [$^\circ$C]', fontsize=16)
#    plt.scatter(y_test_predict,y_test)

    sns.regplot(x=test['SAT_estimate_LGBM'],
                y=test['SAT'],
                scatter_kws={'color': 'white'})

#    plt.title('LightGBM (' + str(mon[mm-1]) + ')', fontsize=20)
    plt.xlabel('Estimated SAT [$^\circ$C]', fontsize=16)
    plt.ylabel('Station SAT [$^\circ$C]', fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
#    plt.scatter(y_test_predict,y_test)
#    plt.text((xmax+xmin)/2.+xint/3.2,(ymax+ymin)/2.-yint/2.0, f"RMSE = {test_rms:.3f}", fontsize=10)
#    plt.text((xmax+xmin)/2.+xint/2.8,(ymax+ymin)/2.-yint/2.2, f"$R^{2} = {test_r2_score:.3f}$", fontsize=10)
    plt.text((mmax+mmin)/2.+mint/3.2,(mmax+mmin)/2.-mint/2.3, f"RMSE = {test_rms:.3f}", fontsize=10)
    plt.text((mmax+mmin)/2.+mint/2.85,(mmax+mmin)/2.-mint/2.5, f"$R^{2} = {test_r2_score:.3f}$", fontsize=10)
    z = np.polyfit(test["SAT_estimate_LGBM"], test["SAT"],1)
    #print(z[0],z[1])
#    if z[1]<0 :
#        plt.text((xmax+xmin)/2.+xint/8.,(ymax+ymin)/2.-yint/8.,f"$y = {z[0]:.3f}x {z[1]:.3f}$", fontsize=10)
#    else:
#        plt.text((xmax+xmin)/2.+xint/8.,(ymax+ymin)/2.-yint/8.,f"$y = {z[0]:.3f}x +{z[1]:.3f}$", fontsize=10)
    if z[1]<0 :
        plt.text((mmax+mmin)/2.+mint/8.,(mmax+mmin)/2.-mint/8.,f"$y = {z[0]:.3f}x {z[1]:.3f}$", fontsize=10)
    else:
        plt.text((mmax+mmin)/2.+mint/8.,(mmax+mmin)/2.-mint/8.,f"$y = {z[0]:.3f}x +{z[1]:.3f}$", fontsize=10)
        
    sns.scatterplot(x='SAT_estimate_LGBM',
                    y='SAT',
#                    hue='YEAR', # different colors by group
#                    style='STN', # different shapes by group
                    s=10, # marker size
                    data=test, legend = False,
                    color='k')
    
    #plt.show()
    plt.savefig(fpath + 'FIG/eps/' + repr(mm) + '_lgbm_reg.eps')
    plt.savefig(fpath + 'FIG/ML/' + repr(mm) + '_lgbm_reg.png')

    
    
#################
#### XGBoost ####
#################
    n_estimators_list = range(1000, 1001, 1)
    history = []

    for n_estimators in n_estimators_list:
    
        XGB = XGBRegressor(n_estimators = n_estimators,  
                                          learning_rate = 0.1)#,
#                                          max_depth = 6)
    
        XGB.fit(X_train, y_train)
        y_train_predict = XGB.predict(X_train)
        y_test_predict = XGB.predict(X_test)

        train_score = (y_train == y_train_predict).mean()
        test_score = (y_test == y_test_predict).mean()

        train_r2_score = r2_score(y_train, y_train_predict)
        test_r2_score = r2_score(y_test, y_test_predict)

        train_rms = sqrt(mean_squared_error(y_train, y_train_predict))
        test_rms = sqrt(mean_squared_error(y_test, y_test_predict))

        importance = XGB.feature_importances_

        print(f"n_estimators = {n_estimators}, train = {train_score:.6f}, test = {test_score:.6f}")

        print(f"train = {train_r2_score:.6f}, test = {test_r2_score:.6f}")
        print(f"train = {train_rms:.6f}, test = {test_rms:.6f}")    
        print(feature_names, importance)

        history.append({
            'n_estimators': n_estimators,
            'train': train_score,
            'test': test_score,
            'variable': feature_names,
            'importance': importance,
        })

# 결과를 판다스의 DataFrame형태로 변형하여 출력합니다.
    history = pd.DataFrame(history)
    history


    train.loc[:,'SAT_estimate_XGB'] = y_train_predict
    test.loc[:,'SAT_estimate_XGB'] = y_test_predict
    train.loc[:,'YEAR'] = train_year
    test.loc[:,'YEAR'] = test_year
    
    train.loc[:,'bias_XGB'] = train['SAT_estimate_XGB'] - train['SAT']
    test.loc[:,'bias_XGB'] = test['SAT_estimate_XGB'] - test['SAT']

#    xint = max(test['SAT_estimate_XGB'])-min(test['SAT_estimate_XGB'])
#    yint = max(test['SAT'])-min(test['SAT'])

#    xmin = min(test['SAT_estimate_XGB'])-xint/10.
#    xmax = max(test['SAT_estimate_XGB'])+xint/10.
#    ymin = min(test['SAT'])-yint/10.
#    ymax = max(test['SAT'])+yint/10.

#    mmin = min(xmin, ymin)
#    mmax = max(xmax, ymax)
#    mint = mmax - mmin

    xint = 25-(-55)
    yint = 25-(-55)

    xmin = -55-xint/10.
    xmax = 25+xint/10.
    ymin = -55-yint/10.
    ymax = 25+yint/10.

    mmin = min(xmin, ymin)
    mmax = max(xmax, ymax)
    mint = mmax - mmin

    plt.figure(figsize = (8, 8))
#    plt.xlim(xmin,xmax)
#    plt.ylim(ymin,ymax)
    plt.xlim(mmin,mmax)
    plt.ylim(mmin,mmax)

#    plt.title('XGBoost ('+ str(mon[mm-1]) + ')', fontsize=20)
    plt.xlabel('Estimated SAT [$^\circ$C]', fontsize=16)
    plt.ylabel('Station SAT [$^\circ$C]', fontsize=16)
#    plt.scatter(y_test_predict,y_test)

    sns.regplot(x=test['SAT_estimate_XGB'],
                y=test['SAT'],
                scatter_kws={'color': 'white'})

#    plt.title('XGBoost (' + str(mon[mm-1]) + ')', fontsize=20)
    plt.xlabel('Estimated SAT [$^\circ$C]', fontsize=16)
    plt.ylabel('Station SAT [$^\circ$C]', fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
#    plt.scatter(y_test_predict,y_test)
#    plt.text((xmax+xmin)/2.+xint/3.2,(ymax+ymin)/2.-yint/2.0, f"RMSE = {test_rms:.3f}", fontsize=10)
#    plt.text((xmax+xmin)/2.+xint/2.8,(ymax+ymin)/2.-yint/2.2, f"$R^{2} = {test_r2_score:.3f}$", fontsize=10)
    plt.text((mmax+mmin)/2.+mint/3.2,(mmax+mmin)/2.-mint/2.3, f"RMSE = {test_rms:.3f}", fontsize=10)
    plt.text((mmax+mmin)/2.+mint/2.85,(mmax+mmin)/2.-mint/2.5, f"$R^{2} = {test_r2_score:.3f}$", fontsize=10)
    z = np.polyfit(test["SAT_estimate_XGB"], test["SAT"],1)
    #print(z[0],z[1])
#    if z[1]<0 :
#        plt.text((xmax+xmin)/2.+xint/8.,(ymax+ymin)/2.-yint/8.,f"$y = {z[0]:.3f}x {z[1]:.3f}$", fontsize=10)
#    else:
#        plt.text((xmax+xmin)/2.+xint/8.,(ymax+ymin)/2.-yint/8.,f"$y = {z[0]:.3f}x +{z[1]:.3f}$", fontsize=10)
    if z[1]<0 :
        plt.text((mmax+mmin)/2.+mint/8.,(mmax+mmin)/2.-mint/8.,f"$y = {z[0]:.3f}x {z[1]:.3f}$", fontsize=10)
    else:
        plt.text((mmax+mmin)/2.+mint/8.,(mmax+mmin)/2.-mint/8.,f"$y = {z[0]:.3f}x +{z[1]:.3f}$", fontsize=10)
        
    sns.scatterplot(x='SAT_estimate_XGB',
                    y='SAT',
#                    hue='YEAR', # different colors by group
#                    style='STN', # different shapes by group
                    s=10, # marker size
                    data=test, legend = False,
                    color='k')

    #plt.show()
    plt.savefig(fpath + 'FIG/eps/' + repr(mm) + '_xg_reg.eps')
    plt.savefig(fpath + 'FIG/ML/' + repr(mm) + '_xg_reg.png')
    

    
    
##### Mean #####
    train.loc[:,'SAT_estimate'] = (train["SAT_estimate_RF"] + train["SAT_estimate_LGBM"] + train["SAT_estimate_XGB"]) / 3.
    test.loc[:,'SAT_estimate'] = (test["SAT_estimate_RF"] + test["SAT_estimate_LGBM"] + test["SAT_estimate_XGB"]) / 3.

    train.loc[:,'bias'] = train['SAT_estimate'] - train['SAT']
    test.loc[:,'bias'] = test['SAT_estimate'] - test['SAT']

    
    y_train_predict = train["SAT_estimate"]
    y_test_predict = test["SAT_estimate"]

    train_score = (y_train == y_train_predict).mean()
    test_score = (y_test == y_test_predict).mean()

    train_r2_score = r2_score(y_train, y_train_predict)
    test_r2_score = r2_score(y_test, y_test_predict)

    train_rms = sqrt(mean_squared_error(y_train, y_train_predict))
    test_rms = sqrt(mean_squared_error(y_test, y_test_predict))


    print(f"train = {train_score:.6f}, test = {test_score:.6f}")
    print(f"train = {train_r2_score:.6f}, test = {test_r2_score:.6f}")
    print(f"train = {train_rms:.6f}, test = {test_rms:.6f}")    

#    history.append({
#        'train': train_score, train_r2_score, train_rms
#        'test': test_score, test_r2_score, test_rms
#    })

# 결과를 판다스의 DataFrame형태로 변형하여 출력합니다.
#    history = pd.DataFrame(history)
#    history


#    xint = max(test['SAT_estimate'])-min(test['SAT_estimate'])
#    yint = max(test['SAT'])-min(test['SAT'])

#    xmin = min(test['SAT_estimate'])-xint/10.
#    xmax = max(test['SAT_estimate'])+xint/10.
#    ymin = min(test['SAT'])-yint/10.
#    ymax = max(test['SAT'])+yint/10.

#    mmin = min(xmin, ymin)
#    mmax = max(xmax, ymax)
#    mint = mmax - mmin

    xint = 25-(-55)
    yint = 25-(-55)

    xmin = -55-xint/10.
    xmax = 25+xint/10.
    ymin = -55-yint/10.
    ymax = 25+yint/10.

    mmin = min(xmin, ymin)
    mmax = max(xmax, ymax)
    mint = mmax - mmin
    
    plt.figure(figsize = (8, 8))
#    plt.xlim(xmin,xmax)
#    plt.ylim(ymin,ymax)
    plt.xlim(mmin,mmax)
    plt.ylim(mmin,mmax)

#    plt.title('Scatter Ploat of Validation ('+ str(mon[mm-1]) + ')', fontsize=20)
    plt.xlabel('Estimated SAT [$^\circ$C]', fontsize=16)
    plt.ylabel('Station SAT [$^\circ$C]', fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
#    plt.scatter(y_test_predict,y_test)

    sns.regplot(x=test['SAT_estimate'],
                y=test['SAT'],
                scatter_kws={'color': 'white'})

#    plt.title('Scatter Ploat of Validation (' + str(mon[mm-1]) + ')', fontsize=20)
    plt.xlabel('Estimated SAT [$^\circ$C]', fontsize=16)
    plt.ylabel('Station SAT [$^\circ$C]', fontsize=16)
#    plt.scatter(y_test_predict,y_test)
#    plt.text((xmax+xmin)/2.+xint/3.2,(ymax+ymin)/2.-yint/2.0, f"RMSE = {test_rms:.3f}", fontsize=10)
#    plt.text((xmax+xmin)/2.+xint/2.8,(ymax+ymin)/2.-yint/2.2, f"$R^{2} = {test_r2_score:.3f}$", fontsize=10)
    plt.text((mmax+mmin)/2.+mint/3.2,(mmax+mmin)/2.-mint/2.3, f"RMSE = {test_rms:.3f}", fontsize=10)
    plt.text((mmax+mmin)/2.+mint/2.85,(mmax+mmin)/2.-mint/2.5, f"$R^{2} = {test_r2_score:.3f}$", fontsize=10)
    z = np.polyfit(test["SAT_estimate"], test["SAT"],1)
    #print(z[0],z[1])
#    if z[1]<0 :
#        plt.text((xmax+xmin)/2.+xint/8.,(ymax+ymin)/2.-yint/8.,f"$y = {z[0]:.3f}x {z[1]:.3f}$", fontsize=10)
#    else:
#        plt.text((xmax+xmin)/2.+xint/8.,(ymax+ymin)/2.-yint/8.,f"$y = {z[0]:.3f}x +{z[1]:.3f}$", fontsize=10)
    if z[1]<0 :
        plt.text((mmax+mmin)/2.+mint/8.,(mmax+mmin)/2.-mint/8.,f"$y = {z[0]:.3f}x {z[1]:.3f}$", fontsize=10)
    else:
        plt.text((mmax+mmin)/2.+mint/8.,(mmax+mmin)/2.-mint/8.,f"$y = {z[0]:.3f}x +{z[1]:.3f}$", fontsize=10)
        
    sns.scatterplot(x='SAT_estimate',
                    y='SAT',
#                    hue='YEAR', # different colors by group
#                    style='STN', # different shapes by group
                    s=10, # marker size
                    data=test, legend = False,
                    color='k')

    #plt.show()
    plt.savefig(fpath + 'FIG/eps/' + repr(mm) + '_reg.eps')
    plt.savefig(fpath + 'FIG/' + repr(mm) + '_reg.png')   
    
    
    
    
    
    plt.figure(figsize = (8, 8))
#    plt.xlim(xmin,xmax)
#    plt.ylim(ymin,ymax)
    plt.xlim(mmin,mmax)
    plt.ylim(mmin,mmax)

#    plt.title('Scatter Ploat of Validation ('+ str(mon[mm-1]) + ')', fontsize=20)
    plt.xlabel('Estimated SAT [$^\circ$C]', fontsize=16)
    plt.ylabel('Station SAT [$^\circ$C]', fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
#    plt.scatter(y_test_predict,y_test)

    sns.regplot(x=test['SAT_estimate'],
                y=test['SAT'],
                scatter_kws={'color': 'white'})

#    plt.title('Scatter Ploat of Validation (' + str(mon[mm-1]) + ')', fontsize=20)
    plt.xlabel('Estimated SAT [$^\circ$C]', fontsize=16) ##fontsize =20
    plt.ylabel('Station SAT [$^\circ$C]', fontsize=16)
#    plt.scatter(y_test_predict,y_test)
#    plt.text((xmax+xmin)/2.+xint/3.2,(ymax+ymin)/2.-yint/2.0, f"RMSE = {test_rms:.3f}", fontsize=15)
#    plt.text((xmax+xmin)/2.+xint/2.8,(ymax+ymin)/2.-yint/2.2, f"$R^{2} = {test_r2_score:.3f}$", fontsize=15)
    plt.text((mmax+mmin)/2.+mint/3.2,(mmax+mmin)/2.-mint/2.3, f"RMSE = {test_rms:.3f}", fontsize=10)
    plt.text((mmax+mmin)/2.+mint/2.85,(mmax+mmin)/2.-mint/2.5, f"$R^{2} = {test_r2_score:.3f}$", fontsize=10)
    z = np.polyfit(test["SAT_estimate"], test["SAT"],1)
    #print(z[0],z[1])
#    if z[1]<0 :
#        plt.text((xmax+xmin)/2.+xint/8.,(ymax+ymin)/2.-yint/8.,f"$y = {z[0]:.3f}x {z[1]:.3f}$", fontsize=15)
#    else:
#        plt.text((xmax+xmin)/2.+xint/8.,(ymax+ymin)/2.-yint/8.,f"$y = {z[0]:.3f}x +{z[1]:.3f}$", fontsize=15)
    if z[1]<0 :
        plt.text((mmax+mmin)/2.+mint/8.,(mmax+mmin)/2.-mint/8.,f"$y = {z[0]:.3f}x {z[1]:.3f}$", fontsize=10)
    else:
        plt.text((mmax+mmin)/2.+mint/8.,(mmax+mmin)/2.-mint/8.,f"$y = {z[0]:.3f}x +{z[1]:.3f}$", fontsize=10)
        
    sns.scatterplot(x='SAT_estimate',
                    y='SAT',
#                    hue='YEAR', # different colors by group
#                    style='STN', # different shapes by group
                    s=10, # marker size
                    data=test, legend = False,
                    color='k')

    #plt.show()
    plt.savefig(repr(mm) + '_reg.png')
   
    
    
    
    train = train[["LAT", "LON", "YEAR", "SAT",
                   "NDVI_t", "LST_DAY_t", "LST_NIGHT_t",
                   "NDVI_a", "LST_DAY_a", "LST_NIGHT_a",
                   "Water", "Evergreen_trees", "Decidous_trees",
                   "Shrub", "Grass", "crops", "Urban", "Snow",
                   "Barren", "Unclassifiend", 
                   "SAT_estimate_RF","SAT_estimate_LGBM", "SAT_estimate_XGB",
                   "SAT_estimate",
                   "bias_RF","bias_LGBM", "bias_XGB",
                   "bias"]]
    
    test = test[["LAT", "LON", "YEAR", "SAT", 
                 "NDVI_t", "LST_DAY_t", "LST_NIGHT_t",
                 "NDVI_a", "LST_DAY_a", "LST_NIGHT_a",
                 "Water", "Evergreen_trees", "Decidous_trees",
                 "Shrub", "Grass", "crops", "Urban", "Snow",
                 "Barren", "Unclassifiend",
                 "SAT_estimate_RF","SAT_estimate_LGBM", "SAT_estimate_XGB",
                 "SAT_estimate",
                 "bias_RF","bias_LGBM", "bias_XGB",
                 "bias"]]

    
    a = train.groupby("STN")
    print(len(a))
    train.to_csv(fpath + repr(mm) + '_train_newest.csv', mode='w',index=True, header=True)
    test.to_csv(fpath + repr(mm) + '_test_newest.csv', mode='w',index=True, header=True)



sorted_idx = model.feature_importances_.argsort()
plt.barh(test.index[sorted_idx], model.feature_importances_[sorted_idx])
plt.xlabel("Random Forest Feature Importance")
